# Level 0 single-seed diagnostics


In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path(os.getenv('NANOGPT_LEVEL0_RESULTS_ROOT','/tmp/nanogpt-level0/results'))
OPT=os.getenv('NANOGPT_LEVEL0_NOTEBOOK_OPTIMIZER','adamw')
SEED=int(os.getenv('NANOGPT_LEVEL0_NOTEBOOK_SEED','1337'))
RUN=ROOT/f'{OPT}_seed_{SEED}'
df=pd.read_csv(RUN/'metrics.csv')
for split in ['train','val','test']: df[f'{split}_error']=1-df[f'{split}_accuracy']
df.head()


In [ ]:
for metric in ['loss','accuracy','error','perplexity']:
    plt.figure(figsize=(9,5))
    for split in ['train','val','test']: plt.plot(df.step,df[f'{split}_{metric}'],label=split)
    plt.xlabel('step'); plt.ylabel(metric); plt.title(f'{OPT} seed {SEED}: {metric}'); plt.legend(); plt.grid(alpha=.25); plt.show()


In [ ]:
files=sorted(RUN.glob('weightwatcher_step_*.csv'))
if files:
    ww=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
    layer_col='layer_id' if 'layer_id' in ww else 'layer'
    for layer,g in ww.groupby(layer_col): plt.plot(g.step,g.alpha,label=str(layer))
    plt.xlabel('step'); plt.ylabel('alpha'); plt.legend(bbox_to_anchor=(1.02,1)); plt.show()
else: print('No WeightWatcher files found.')
